**`Instalação das Dependências`**

In [1]:
!pip install arxiv pandas tqdm nltk

**`Extração + NLP + Filtros`**



In [6]:
import arxiv
import pandas as pd
import re
from datetime import datetime, timezone
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ==========================================
# 1. CONFIGURAÇÃO DO AMBIENTE DE NLP (NLTK)
# ==========================================
print("Configurando ambiente de NLP...")
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

stop_words = set(stopwords.words('english')) # A API do arXiv publica em inglês
lemmatizer = WordNetLemmatizer()

def pipeline_limpeza_texto(texto):
    """
    Executa a limpeza de NLP: remove LaTeX/caracteres especiais,
    converte para lowercase, remove stopwords e aplica lematização.
    """
    if not texto:
        return ""

    # A. Remover comandos LaTeX comuns e marcações de equações ($...$)
    texto = re.sub(r'\$.*?\$', '', texto)

    # B. Remover quebras de linha (\n), tabulações e espaços excessivos
    texto = texto.replace("\n", " ").replace("\t", " ")

    # C. Remover caracteres especiais e pontuações (mantendo letras e números)
    texto = re.sub(r'[^a-zA-Z0-9\s]', '', texto)

    # D. Converter para minúsculas (lower case) e quebrar em palavras
    palavras = texto.lower().split()

    # E. Remover Stopwords e aplicar Lemmatization (Reduzir à palavra raiz)
    palavras_processadas = [
        lemmatizer.lemmatize(palavra)
        for palavra in palavras
        if palavra not in stop_words
    ]

    # Reconstrói a string tratada
    return " ".join(palavras_processadas)

# ==========================================
# 2. MAPEAMENTO E EXTRAÇÃO POR CATEGORIA E ANO (FILTRO NATIVO)
# ==========================================
categorias_alvo = {
    "Inteligencia artificial": "cs.AI",
    "Data Science": "cs.LG",
    "Cloud Computing": '("cloud computing" OR "distributed systems")',
    "Devops": '("devops" OR "continuous integration" OR "software deployment")',
    "Backend": '("backend" OR "web server" OR "microservices" OR "api development")',
    "Frontend": '("frontend" OR "user interface" OR "web application" OR "javascript framework")',
    "Banco de Dados": "cs.DB",
    "Cibersegurança": "cs.CR",
    "Redes": "cs.NI",
    "Mobile": '("mobile computing" OR "android" OR "ios" OR "mobile application")'
}

# Definição dos anos para a construção da linha do tempo do KES
anos_alvo = [2021, 2022, 2023, 2024, 2025, 2026]

client = arxiv.Client()
todos_artigos = []
ids_coletados = set() # Evita duplicidade cruzada entre buscas

print("\nIniciando extração do arXiv (100 artigos por CATEGORIA por ANO)...")

for nome_categoria, termo_busca in categorias_alvo.items():
    print(f"\nProcessando Categoria: {nome_categoria}")

    for ano in anos_alvo:
        # Inserção do filtro temporal nativo na query enviado para a API do arXiv
        # Formato exigido pelo arXiv: [ANO01010000 TO ANO12312359]
        query_temporal = f"{termo_busca} AND submittedDate:[{ano}01010000 TO {ano}12312359]"

        search = arxiv.Search(
            query=query_temporal,
            max_results=150, # Margem suficiente já que a API filtra nativamente pelo ano exato
            sort_by=arxiv.SortCriterion.SubmittedDate
        )

        contador_ano_categoria = 0
        try:
            for result in client.results(search):
                if contador_ano_categoria >= 100:
                    break

                id_arxiv = result.entry_id.split("/abs/")[-1]

                if id_arxiv not in ids_coletados:
                    ids_coletados.add(id_arxiv)
                    contador_ano_categoria += 1

                    todos_artigos.append({
                        "titulo": result.title,
                        "resumo_original": result.summary,
                        "autores": ", ".join([author.name for author in result.authors]),
                        "ano": ano,
                        "link": result.pdf_url,
                        "categoria_projeto": nome_categoria
                    })
        except Exception as e:
            print(f"  <!> Erro ou limite excedido ao buscar o ano {ano}: {e}")

        print(f"  -> Ano {ano}: {contador_ano_categoria} artigos coletados.")

# Criar DataFrame Inicial
df = pd.DataFrame(todos_artigos)

# ==========================================
# 3. TRATAMENTO DE NULOS, OUTLIERS E NLP
# ==========================================
print("\nIniciando o tratamento de dados e NLP...")

# A. Remover nulos caso existam
df = df.dropna(subset=["titulo", "resumo_original"])

# B. Aplicar a função de Limpeza de Texto (Criação de nova feature tratada)
df['resumo_limpo'] = df['resumo_original'].apply(pipeline_limpeza_texto)

# C. Filtrar Outliers: Remover resumos vazios ou muito curtos (menos de 3 palavras pós-limpeza)
df['tamanho_resumo_limpo'] = df['resumo_limpo'].apply(lambda x: len(x.split()))
df = df[df['tamanho_resumo_limpo'] >= 3]

# D. Remover coluna temporária de contagem
df = df.drop(columns=['tamanho_resumo_limpo'])

# ==========================================
# 4. CRIAÇÃO DO ID ÚNICO E EXPORTAÇÃO
# ==========================================
# Reinicia o índice para gerar um ID limpo, incremental e sequencial (1, 2, 3...)
df = df.reset_index(drop=True)
df.index = df.index + 1
df.index.name = 'id'
df_final = df.reset_index()

# Exibir relatórios de validação no console
print("\n================ DATASET PRONTO PARA A LINHA DO TEMPO ================")
print(f"Total Geral de Artigos no Dataset: {len(df_final)}")
print("\nDistribuição Homogênea por Ano:")
print(df_final["ano"].value_counts().sort_index())
print("\nDistribuição por Categoria Global:")
print(df_final["categoria_projeto"].value_counts())
print("====================================================================\n")

# Visualizar amostra rápida das primeiras linhas
print("Amostra dos dados processados temporalmente:")
print(df_final[['id', 'ano', 'categoria_projeto', 'titulo']].head(3))

# Exportação dos artefatos finais para integração com o time
df_final.to_json("dataset_techmind_evolution.json", orient="records", force_ascii=False, indent=4)
df_final.to_csv("dataset_techmind_evolution.csv", index=False, encoding="utf-8")

print("\nSucesso! Arquivos 'dataset_techmind_evolution.json' e 'dataset_techmind_evolution.csv' gerados.")

Configurando ambiente de NLP...

Iniciando extração do arXiv (100 artigos por CATEGORIA por ANO)...

Processando Categoria: Inteligencia artificial
  -> Ano 2021: 100 artigos coletados.
  -> Ano 2022: 100 artigos coletados.
  -> Ano 2023: 100 artigos coletados.
  -> Ano 2024: 100 artigos coletados.
  -> Ano 2025: 100 artigos coletados.
  -> Ano 2026: 100 artigos coletados.

Processando Categoria: Data Science
  -> Ano 2021: 98 artigos coletados.
  -> Ano 2022: 100 artigos coletados.
  -> Ano 2023: 100 artigos coletados.
  -> Ano 2024: 100 artigos coletados.
  -> Ano 2025: 100 artigos coletados.
  -> Ano 2026: 100 artigos coletados.

Processando Categoria: Cloud Computing
  -> Ano 2021: 100 artigos coletados.
  -> Ano 2022: 100 artigos coletados.
  -> Ano 2023: 100 artigos coletados.
  -> Ano 2024: 100 artigos coletados.
  -> Ano 2025: 100 artigos coletados.
  -> Ano 2026: 100 artigos coletados.

Processando Categoria: Devops
  -> Ano 2021: 80 artigos coletados.
  -> Ano 2022: 92 artigo

**`Verificação de artigos duplicados`**

In [7]:
# Verifica se existem IDs do arXiv duplicados no DataFrame final
duplicados = df_final['link'].duplicated().sum()
print(f"Total de artigos duplicados encontrados: {duplicados}")

Total de artigos duplicados encontrados: 0


**`Download dataset gerado em json e csv`**

In [9]:
from google.colab import files

# Baixar o arquivo JSON
files.download('dataset_techmind_evolution.json')

# Baixar o arquivo CSV
files.download('dataset_techmind_evolution.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>